# MUST-Former Training Pipeline

This notebook contains the complete implementation for training the MUST-Former family of models on the Sen1Floods11 hand-labeled dataset. The workflow includes:

- **Data Preprocessing:** Processing of Sentinel-1 (SAR) and Sentinel-2 (Optical) imagery, including normalization and the computation of spectral indices (NDVI, NDWI).
- **Hybrid Loss Function:** Implementation of a custom loss function combining Focal Cross-Entropy and Dice Loss to effectively handle class imbalance.
- **Regional Weighting:** A dynamic weighting mechanism that assigns higher training importance to geographic regions with greater flood water content.
- **Model Architectures:** Definitions for all five MUST-Former variants: MUST-Former-SAR, MUST-Former-Optical, MUST-Former-Projector,MUST-Former-CrossAttn , and MUST-Former-PCA .
- **Training Pipeline:** The complete training configuration, optimization loop, and validation metrics used to train and evaluate the models.

In [1]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import subprocess
import sys

packages = [
    "torch torchvision torchaudio",
    "transformers rasterio albumentations",
    "pandas matplotlib scikit-learn"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + package.split())

In [21]:
import os
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import rasterio
import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


class PathConfig:
    S1_HAND = "/content/drive/MyDrive/Sen1Floods11/data/flood_events/HandLabeled/S1Hand"
    S2_HAND = "/content/drive/MyDrive/Sen1Floods11/data/flood_events/HandLabeled/S2Hand"
    LABEL_HAND = "/content/drive/MyDrive/Sen1Floods11/data/flood_events/HandLabeled/LabelHand"
    JRC_WATER = "/content/drive/MyDrive/Sen1Floods11/data/flood_events/HandLabeled/JRCWaterHand"

    TRAIN_SPLIT = "/content/drive/MyDrive/Sen1Floods11/splits/flood_handlabeled/flood_train_data.csv"
    VAL_SPLIT = "/content/drive/MyDrive/Sen1Floods11/splits/flood_handlabeled/flood_valid_data.csv"
    TEST_SPLIT = "/content/drive/MyDrive/Sen1Floods11/splits/flood_handlabeled/flood_test_data.csv"
    BOLIVIA_SPLIT = "/content/drive/MyDrive/Sen1Floods11/splits/flood_handlabeled/flood_bolivia_data.csv"

    METADATA = "/content/drive/MyDrive/Sen1Floods11/data/Sen1Floods11_Metadata.geojson"
    OUTPUT_DIR = "/content/drive/MyDrive/Sen1Floods11/5_SegformerModelsOutputs"

    def create_dirs(self):
        os.makedirs(self.OUTPUT_DIR, exist_ok=True)
        os.makedirs(os.path.join(self.OUTPUT_DIR, 'checkpoints'), exist_ok=True)
        os.makedirs(os.path.join(self.OUTPUT_DIR, 'visualizations'), exist_ok=True)


paths = PathConfig()
paths.create_dirs()


def load_and_process_metadata(metadata_path):
    """Load the region metadata geojson and compute the center coordinates of each region."""
    with open(metadata_path, 'r') as f:
        geojson_data = json.load(f)

    metadata_list = []
    for feature in geojson_data['features']:
        props = feature['properties']
        region_id = props.get('location', str(props.get('ID', 'unknown')))
        coords = feature['geometry']['coordinates'][0]
        lons = [c[0] for c in coords]
        lats = [c[1] for c in coords]
        props['region_id'] = str(region_id)
        props['center_lon'] = (min(lons) + max(lons)) / 2
        props['center_lat'] = (min(lats) + max(lats)) / 2
        metadata_list.append(props)

    metadata_df = pd.DataFrame(metadata_list)
    print(f"Loaded metadata for {len(metadata_df)} regions")
    return metadata_df


metadata = load_and_process_metadata(paths.METADATA)


def create_file_mapping(base_dir):
    """Walk a directory and map each tif filename to its full path."""
    mapping = {}
    if not os.path.exists(base_dir):
        print(f"Warning: directory does not exist: {base_dir}")
        return mapping

    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.tif'):
                full_path = os.path.join(root, file)
                if file in mapping:
                    raise RuntimeError(f"Duplicate filename detected: {file}")
                mapping[file] = full_path
    return mapping


s1_mapping = create_file_mapping(paths.S1_HAND)
s2_mapping = create_file_mapping(paths.S2_HAND)
label_mapping = create_file_mapping(paths.LABEL_HAND)
jrc_mapping = create_file_mapping(paths.JRC_WATER)

print(f"S1: {len(s1_mapping)} files")
print(f"S2: {len(s2_mapping)} files")
print(f"Labels: {len(label_mapping)} files")
print(f"JRC: {len(jrc_mapping)} files")


def load_and_process_split(split_path, split_name):
    """Load a split csv and tag each row with the region it belongs to."""
    if not os.path.exists(split_path):
        print(f"Split file not found: {split_path}")
        return None

    df = pd.read_csv(split_path)
    s1_col, label_col = df.columns[0], df.columns[1]
    df = df.rename(columns={s1_col: 's1_filename', label_col: 'label'})

    region_mapping = {
        'Bolivia': 'Bolivia', 'Colombia': 'Colombia', 'Ghana': 'Ghana',
        'India': 'India', 'Cambodia': 'Cambodia', 'Mekong': 'Cambodia',
        'Nigeria': 'Nigeria', 'Pakistan': 'Pakistan', 'Paraguay': 'Paraguay',
        'Somalia': 'Somalia', 'Spain': 'Spain', 'Sri-Lanka': 'Sri-Lanka',
        'Sri Lanka': 'Sri-Lanka', 'USA': 'USA'
    }

    def extract_region_from_filename(filename):
        if pd.isna(filename):
            return 'unknown'
        filename_str = str(filename).strip()
        for prefix, region in region_mapping.items():
            if filename_str.startswith(prefix):
                return region
        return 'unknown'

    df['region_id'] = df['label'].apply(extract_region_from_filename).astype(str)
    df['s1_filename'] = df['s1_filename'].astype(str)
    df['label'] = df['label'].astype(str)

    print(f"{split_name}: {len(df)} samples, {df['region_id'].nunique()} regions")
    return df


train_df = load_and_process_split(paths.TRAIN_SPLIT, "Training")
val_df = load_and_process_split(paths.VAL_SPLIT, "Validation")
test_df = load_and_process_split(paths.TEST_SPLIT, "Test")

print(f"Training: {len(train_df)} samples")
print(f"Validation: {len(val_df)} samples")
print(f"Test: {len(test_df)} samples")

assert len(train_df) > 0, "Training data is empty"
assert len(val_df) > 0, "Validation data is empty"
assert len(test_df) > 0, "Test data is empty"

In [20]:
def load_sample_tiff(path):
    """Read a tif file into a numpy array."""
    try:
        with rasterio.open(path) as src:
            return src.read()
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None


def compute_regional_water_statistics(df, split_name):
    """
    Compute per region pixel counts for a split: total, water, background,
    ignored pixels, and the permanent water vs flood water split using JRC.
    """
    print(f"Computing water statistics for {split_name}")
    regional_stats = []

    for region_id in sorted(df['region_id'].unique()):
        region_df = df[df['region_id'] == region_id]
        total_pixels = water_pixels = background_pixels = ignore_pixels = 0
        permanent_water_pixels = flood_pixels = 0

        for _, row in region_df.iterrows():
            label_filename = row['label']
            label_path = label_mapping.get(label_filename)
            jrc_filename = label_filename.replace('LabelHand', 'JRCWaterHand')
            jrc_path = jrc_mapping.get(jrc_filename)

            if not label_path:
                continue

            label = load_sample_tiff(label_path)
            if label is None:
                continue
            label = label[0]

            valid_vals = {-1, 0, 1, 255}
            unique_vals = np.unique(label)
            if not all(v in valid_vals for v in unique_vals):
                mask_invalid = ~np.isin(label, [-1, 0, 1, 255])
                label = np.where(mask_invalid, 255, label)

            total_pixels += label.size
            water_pixels += (label == 1).sum()
            background_pixels += (label == 0).sum()
            ignore_pixels += ((label == -1) | (label == 255)).sum()

            if jrc_path:
                jrc = load_sample_tiff(jrc_path)
                if jrc is not None:
                    jrc = jrc[0]
                    permanent_water_pixels += ((jrc == 1) & (label == 1)).sum()
                    flood_pixels += ((jrc == 0) & (label == 1)).sum()

        valid_pixels = max(1, total_pixels - ignore_pixels)
        regional_stats.append({
            'region_id': region_id,
            'split': split_name,
            'total_samples': len(region_df),
            'total_pixels': total_pixels,
            'valid_pixels': valid_pixels,
            'water_pixels': water_pixels,
            'background_pixels': background_pixels,
            'permanent_water_pixels': permanent_water_pixels,
            'flood_pixels': flood_pixels,
            'water_ratio': water_pixels / valid_pixels,
            'flood_ratio': flood_pixels / valid_pixels,
            'permanent_water_ratio': permanent_water_pixels / valid_pixels
        })

    return pd.DataFrame(regional_stats)


train_stats = compute_regional_water_statistics(train_df, 'train')
val_stats = compute_regional_water_statistics(val_df, 'val')
test_stats = compute_regional_water_statistics(test_df, 'test')

all_stats = pd.concat([train_stats, val_stats, test_stats], ignore_index=True)
stats_path = os.path.join(paths.OUTPUT_DIR, 'region_water_stats.csv')
all_stats.to_csv(stats_path, index=False)
print(f"Regional statistics saved to: {stats_path}")

print(f"{'Region':<15} {'Samples':<8} {'Water%':<8} {'Flood%':<8} {'Perm%':<8}")
for _, row in train_stats.sort_values('water_ratio', ascending=False).iterrows():
    print(f"{row['region_id']:<15} {row['total_samples']:<8} "
          f"{row['water_ratio']*100:<8.2f} {row['flood_ratio']*100:<8.2f} "
          f"{row['permanent_water_ratio']*100:<8.2f}")

In [5]:
from transformers import SegformerForSemanticSegmentation, SegformerConfig


class MultiModalSegFormer(nn.Module):
    """
    Multi-modal SegFormer for SAR-optical fusion.
    Supports three fusion strategies: projector, attention, and pca.
    """
    def __init__(self, backbone='nvidia/mit-b2', num_classes=2, s1_channels=2,
                 s2_channels=13, include_s1=True, include_s2=True,
                 fusion_type='projector', pretrained=True, dropout=0.1):
        super().__init__()
        self.include_s1 = include_s1
        self.include_s2 = include_s2
        self.fusion_type = fusion_type
        self.s1_channels = s1_channels if include_s1 else 0
        self.s2_channels = s2_channels if include_s2 else 0
        self.extra_channels = 2 if include_s2 else 0  # ndvi, ndwi

        if include_s1 and include_s2:
            self.total_input_channels = self.s1_channels + self.s2_channels + self.extra_channels
        elif include_s1:
            self.total_input_channels = self.s1_channels
        elif include_s2:
            self.total_input_channels = self.s2_channels + self.extra_channels
        else:
            raise ValueError("At least one of include_s1 or include_s2 must be True")

        if fusion_type == 'projector':
            self._init_projector_fusion()
        elif fusion_type == 'attention':
            self._init_attention_fusion()
        elif fusion_type == 'pca':
            self._init_pca_fusion()
        else:
            raise ValueError(f"Unknown fusion_type: {fusion_type}")

        if pretrained:
            try:
                self.segformer = SegformerForSemanticSegmentation.from_pretrained(
                    backbone, num_labels=num_classes, ignore_mismatched_sizes=True
                )
            except Exception:
                self.segformer = SegformerForSemanticSegmentation(
                    SegformerConfig(num_labels=num_classes)
                )
        else:
            self.segformer = SegformerForSemanticSegmentation(
                SegformerConfig(num_labels=num_classes)
            )

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def _init_projector_fusion(self):
        """Simple 1x1 conv projection of the input channels down to 3 channels."""
        if self.include_s1 and not self.include_s2:
            self.s1_proj = nn.Sequential(nn.Conv2d(self.s1_channels, 3, 1), nn.BatchNorm2d(3))
            nn.init.xavier_uniform_(self.s1_proj[0].weight, gain=0.02)
            nn.init.zeros_(self.s1_proj[0].bias)
        elif self.include_s2 and not self.include_s1:
            self.s2_proj = nn.Sequential(nn.Conv2d(self.s2_channels + self.extra_channels, 3, 1), nn.BatchNorm2d(3))
            nn.init.xavier_uniform_(self.s2_proj[0].weight, gain=0.02)
            nn.init.zeros_(self.s2_proj[0].bias)
        else:
            self.s1_proj = nn.Sequential(nn.Conv2d(self.s1_channels, 3, 1), nn.BatchNorm2d(3))
            self.s2_proj = nn.Sequential(nn.Conv2d(self.s2_channels + self.extra_channels, 3, 1), nn.BatchNorm2d(3))
            nn.init.xavier_uniform_(self.s1_proj[0].weight, gain=0.02)
            nn.init.zeros_(self.s1_proj[0].bias)
            nn.init.xavier_uniform_(self.s2_proj[0].weight, gain=0.02)
            nn.init.zeros_(self.s2_proj[0].bias)

    def _init_attention_fusion(self):
        """Cross-attention fusion: project both modalities, downsample spatially, cross attend, fuse."""
        proj_dim = 128
        num_heads = 4
        spatial_reduction = 16

        if self.include_s1:
            self.s1_proj = nn.Sequential(nn.Conv2d(self.s1_channels, proj_dim, 1), nn.BatchNorm2d(proj_dim), nn.GELU())
            nn.init.xavier_uniform_(self.s1_proj[0].weight, gain=0.02)
            nn.init.zeros_(self.s1_proj[0].bias)

        if self.include_s2:
            self.s2_proj = nn.Sequential(nn.Conv2d(self.s2_channels + self.extra_channels, proj_dim, 1), nn.BatchNorm2d(proj_dim), nn.GELU())
            nn.init.xavier_uniform_(self.s2_proj[0].weight, gain=0.02)
            nn.init.zeros_(self.s2_proj[0].bias)

        self.spatial_reduction = spatial_reduction
        if spatial_reduction > 1:
            self.downsample = nn.Sequential(
                nn.Conv2d(proj_dim, proj_dim, spatial_reduction, stride=spatial_reduction, groups=proj_dim),
                nn.BatchNorm2d(proj_dim)
            )
            nn.init.xavier_uniform_(self.downsample[0].weight, gain=0.02)

        self.fusion_norm1 = nn.LayerNorm(proj_dim)
        self.fusion_norm2 = nn.LayerNorm(proj_dim)
        self.cross_attn_12 = nn.MultiheadAttention(proj_dim, num_heads, dropout=0.0, batch_first=True)
        self.cross_attn_21 = nn.MultiheadAttention(proj_dim, num_heads, dropout=0.0, batch_first=True)

        self.ffn1 = nn.Sequential(
            nn.Linear(proj_dim, int(proj_dim * 1.5)), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(int(proj_dim * 1.5), proj_dim), nn.Dropout(0.1)
        )
        self.ffn2 = nn.Sequential(
            nn.Linear(proj_dim, int(proj_dim * 1.5)), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(int(proj_dim * 1.5), proj_dim), nn.Dropout(0.1)
        )

        for module in [self.ffn1, self.ffn2]:
            for layer in module:
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight, gain=0.02)
                    nn.init.zeros_(layer.bias)

        self.norm_ffn1 = nn.LayerNorm(proj_dim)
        self.norm_ffn2 = nn.LayerNorm(proj_dim)

        self.fusion_proj = nn.Sequential(
            nn.Conv2d(proj_dim * 2, proj_dim, 1), nn.BatchNorm2d(proj_dim), nn.GELU(),
            nn.Conv2d(proj_dim, 3, 1), nn.BatchNorm2d(3)
        )
        self.single_proj = nn.Sequential(nn.Conv2d(proj_dim, 3, 1), nn.BatchNorm2d(3))

        nn.init.xavier_uniform_(self.fusion_proj[0].weight, gain=0.02)
        nn.init.zeros_(self.fusion_proj[0].bias)
        nn.init.xavier_uniform_(self.fusion_proj[3].weight, gain=0.02)
        nn.init.zeros_(self.fusion_proj[3].bias)
        nn.init.xavier_uniform_(self.single_proj[0].weight, gain=0.02)
        nn.init.zeros_(self.single_proj[0].bias)

    def _init_pca_fusion(self):
        """Channel-wise standardization followed by a 1x1 conv down to 3 channels."""
        self.pca_proj = nn.Sequential(nn.Conv2d(self.total_input_channels, 3, 1), nn.BatchNorm2d(3))
        nn.init.xavier_uniform_(self.pca_proj[0].weight, gain=0.02)
        nn.init.zeros_(self.pca_proj[0].bias)

    def forward(self, x, metadata=None):
        B, C, H, W = x.shape

        expected = (self.s1_channels + self.s2_channels + self.extra_channels) if \
                   (self.include_s1 and self.include_s2) else \
                   (self.s1_channels or (self.s2_channels + self.extra_channels))
        assert C == expected, f"Channel mismatch: got {C}, expected {expected}"

        if self.fusion_type == 'projector':
            x_fused = self._forward_projector(x)
        elif self.fusion_type == 'attention':
            x_fused = self._forward_attention(x)
        else:
            x_fused = self._forward_pca(x)

        x_fused = self.dropout(x_fused)
        outputs = self.segformer(pixel_values=x_fused)
        logits = outputs.logits
        upsampled = nn.functional.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
        return upsampled

    def _forward_projector(self, x):
        if self.include_s1 and not self.include_s2:
            return self.s1_proj(x)
        elif self.include_s2 and not self.include_s1:
            return self.s2_proj(x)
        else:
            s1_out = self.s1_proj(x[:, :self.s1_channels])
            s2_out = self.s2_proj(x[:, self.s1_channels:])
            return s1_out + s2_out

    def _forward_attention(self, x):
        features = []
        if self.include_s1:
            features.append(self.s1_proj(x[:, :self.s1_channels]))
        if self.include_s2:
            s2_start = self.s1_channels if self.include_s1 else 0
            s2_end = s2_start + self.s2_channels + self.extra_channels
            features.append(self.s2_proj(x[:, s2_start:s2_end]))

        if len(features) == 2:
            f1, f2 = features
            B_, C_, H_, W_ = f1.shape

            if getattr(self, 'spatial_reduction', 1) > 1:
                f1d = self.downsample(f1)
                f2d = self.downsample(f2)
                _, _, Hd, Wd = f1d.shape
            else:
                f1d, f2d, Hd, Wd = f1, f2, H_, W_

            f1_tokens = f1d.flatten(2).permute(0, 2, 1)
            f2_tokens = f2d.flatten(2).permute(0, 2, 1)

            f1n = self.fusion_norm1(f1_tokens)
            f2n = self.fusion_norm2(f2_tokens)
            f1n = f1n / (f1n.std(dim=-1, keepdim=True) + 1e-8)
            f2n = f2n / (f2n.std(dim=-1, keepdim=True) + 1e-8)

            attn1, _ = self.cross_attn_12(f1n, f2n, f2n)
            attn1 = attn1 + f1_tokens
            attn1 = attn1 + self.ffn1(self.norm_ffn1(attn1))

            attn2, _ = self.cross_attn_21(f2n, f1n, f1n)
            attn2 = attn2 + f2_tokens
            attn2 = attn2 + self.ffn2(self.norm_ffn2(attn2))

            attn1 = attn1.permute(0, 2, 1).reshape(B_, C_, Hd, Wd)
            attn2 = attn2.permute(0, 2, 1).reshape(B_, C_, Hd, Wd)

            if getattr(self, 'spatial_reduction', 1) > 1:
                attn1 = nn.functional.interpolate(attn1, size=(H_, W_), mode='bilinear', align_corners=False)
                attn2 = nn.functional.interpolate(attn2, size=(H_, W_), mode='bilinear', align_corners=False)

            return self.fusion_proj(torch.cat([attn1, attn2], dim=1))
        else:
            return self.single_proj(features[0])

    def _forward_pca(self, x):
        x_mean = x.mean(dim=1, keepdim=True)
        x_std = x.std(dim=1, keepdim=True) + 1e-8
        return self.pca_proj((x - x_mean) / x_std)


def create_model(model_type='fusion', backbone='nvidia/mit-b2', fusion_type='projector', pretrained=True, num_classes=2):
    """Build a model variant by name: fusion, s1_only, or s2_only."""
    include_s1 = model_type in ['fusion', 's1_only', 'sar1']
    include_s2 = model_type in ['fusion', 's2_only', 'sar2']

    return MultiModalSegFormer(
        backbone=backbone,
        num_classes=num_classes,
        s1_channels=2,
        s2_channels=13,
        include_s1=include_s1,
        include_s2=include_s2,
        fusion_type=fusion_type,
        pretrained=pretrained
    )


MODEL_CONFIGS = {
    'fusion_projector': {'model_type': 'fusion', 'fusion_type': 'projector'},
    'fusion_attention': {'model_type': 'fusion', 'fusion_type': 'attention'},
    'fusion_pca': {'model_type': 'fusion', 'fusion_type': 'pca'},
    's1_only': {'model_type': 's1_only', 'fusion_type': 'projector'},
    's2_only': {'model_type': 's2_only', 'fusion_type': 'projector'},
}

In [6]:
class Sen1Floods11Dataset(Dataset):
    """
    Dataset for Sen1Floods11. Loads S1 (VV, VH) and S2 (13 bands) tiles,
    normalizes them, and appends NDVI and NDWI computed from S2 bands.

    NDVI = (NIR - Red) / (NIR + Red)
    NDWI = (Green - NIR) / (Green + NIR)

    Sentinel-2 band indices used: Green = band 3 (index 2), Red = band 4 (index 3), NIR = band 8 (index 7).
    """
    def __init__(self, df, s1_mapping, s2_mapping, label_mapping, jrc_mapping,
                 transform=None, include_s1=True, include_s2=True):
        self.df = df.reset_index(drop=True)
        self.s1_mapping = s1_mapping
        self.s2_mapping = s2_mapping
        self.label_mapping = label_mapping
        self.jrc_mapping = jrc_mapping
        self.transform = transform
        self.include_s1 = include_s1
        self.include_s2 = include_s2

    def load_tiff(self, path):
        """Read a tif file, replacing nan/inf values with 0."""
        try:
            with rasterio.open(path) as src:
                data = src.read()
                data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
                return data
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return None

    def compute_indices(self, s2_data):
        """Compute NDVI and NDWI from raw Sentinel-2 reflectance bands."""
        green = s2_data[2].astype(np.float32)
        red = s2_data[3].astype(np.float32)
        nir = s2_data[7].astype(np.float32)
        eps = 1e-8

        ndvi = (nir - red) / (nir + red + eps)
        ndvi = np.nan_to_num(np.clip(ndvi, -1, 1), nan=0.0)

        ndwi = (green - nir) / (green + nir + eps)
        ndwi = np.nan_to_num(np.clip(ndwi, -1, 1), nan=0.0)

        return ndvi, ndwi

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        s1_filename = row.iloc[0]
        label_filename = row.iloc[1]
        s2_filename = label_filename.replace('LabelHand', 'S2Hand')

        s1_path = self.s1_mapping.get(s1_filename)
        s2_path = self.s2_mapping.get(s2_filename)
        label_path = self.label_mapping.get(label_filename)

        s1 = self.load_tiff(s1_path) if s1_path and self.include_s1 else None
        s2 = self.load_tiff(s2_path) if s2_path and self.include_s2 else None
        label = self.load_tiff(label_path)

        if label is None:
            raise ValueError(f"Could not load label for {label_filename}")
        label = label[0].astype(np.int64)

        mask_invalid = ~np.isin(label, [-1, 0, 1, 255])
        if mask_invalid.any():
            label = np.where(mask_invalid, 255, label)
        label = np.where(label == -1, 255, label)

        channels = []

        if self.include_s1 and s1 is not None:
            s1_vv = s1[0].astype(np.float32)
            s1_vh = s1[1].astype(np.float32)

            s1_vv = np.nan_to_num(s1_vv, nan=-25.0, posinf=0.0, neginf=-50.0)
            s1_vh = np.nan_to_num(s1_vh, nan=-25.0, posinf=0.0, neginf=-50.0)
            s1_vv = np.clip(s1_vv, -50, 0)
            s1_vh = np.clip(s1_vh, -50, 0)
            s1_vv = (s1_vv + 50) / 50.0
            s1_vh = (s1_vh + 50) / 50.0

            channels.extend([s1_vv, s1_vh])

        if self.include_s2 and s2 is not None:
            s2f = s2.astype(np.float32)
            s2f = np.nan_to_num(s2f, nan=0.0, posinf=10000.0, neginf=0.0)
            s2n = np.clip(s2f / 10000.0, 0, 1)

            for i in range(s2n.shape[0]):
                channels.append(s2n[i])

            ndvi, ndwi = self.compute_indices(s2f)
            channels.extend([ndvi, ndwi])

        if len(channels) == 0:
            raise ValueError("No input channels available")

        image = np.stack(channels, axis=-1)

        if np.isnan(image).any() or np.isinf(image).any():
            image = np.nan_to_num(image, nan=0.0, posinf=1.0, neginf=0.0)

        if self.transform:
            augmented = self.transform(image=image, mask=label)
            image = augmented['image']
            label = augmented['mask']
            label = label.long() if isinstance(label, torch.Tensor) else torch.from_numpy(label).long()
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float()
            label = torch.from_numpy(label).long()

        if torch.isnan(image).any() or torch.isinf(image).any():
            image = torch.nan_to_num(image, nan=0.0, posinf=1.0, neginf=0.0)
        if label.min() < 0 or label.max() > 255:
            label = torch.clamp(label, 0, 255)

        region_id = str(label_filename).split('_')[0]
        meta = {'region_id': region_id, 'filename': label_filename}

        return image, label, meta


def get_train_transforms():
    """Augmentation pipeline used for training."""
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(scale=(0.9, 1.1), translate_percent=(-0.1, 0.1), rotate=(-15, 15), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.ElasticTransform(alpha=1, sigma=50, p=0.2),
        ToTensorV2()
    ])


def get_val_transforms():
    """No augmentation, used for validation and test."""
    return A.Compose([ToTensorV2()])


def create_dataloaders(train_df, val_df, test_df, batch_size=4, include_s1=True, include_s2=True):
    """Build train, validation and test DataLoaders."""
    train_dataset = Sen1Floods11Dataset(
        train_df, s1_mapping, s2_mapping, label_mapping, jrc_mapping,
        get_train_transforms(), include_s1, include_s2
    )
    val_dataset = Sen1Floods11Dataset(
        val_df, s1_mapping, s2_mapping, label_mapping, jrc_mapping,
        get_val_transforms(), include_s1, include_s2
    )
    test_dataset = Sen1Floods11Dataset(
        test_df, s1_mapping, s2_mapping, label_mapping, jrc_mapping,
        get_val_transforms(), include_s1, include_s2
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                               num_workers=2, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    print(f"Dataloaders created: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")
    return train_loader, val_loader, test_loader

In [7]:
import torch.nn.functional as F


class SegmentationMetrics:
    """Compute per class IoU for binary segmentation."""
    def __init__(self, num_classes=2, ignore_index=255):
        self.num_classes = num_classes
        self.ignore_index = ignore_index

    @torch.no_grad()
    def iou_per_class(self, logits, target):
        """Return a list of IoU scores, one per class, ignoring the ignore_index."""
        pred = logits.argmax(dim=1)
        mask = (target != self.ignore_index)
        ious = []

        for c in range(self.num_classes):
            pred_c = (pred == c) & mask
            targ_c = (target == c) & mask
            inter = (pred_c & targ_c).sum().item()
            union = (pred_c | targ_c).sum().item()
            ious.append(inter / (union + 1e-8))

        return ious


class HybridLoss(nn.Module):
    """
    Combines focal cross entropy with dice loss.

    Loss = (1 - dice_weight) * FocalCE + dice_weight * DiceLoss

    Focal loss: FL = -alpha * (1 - p_t)^gamma * log(p_t)
    Down-weights easy examples so the model focuses on hard pixels.
    alpha sets the class weight (higher for the minority water class),
    gamma controls how strongly easy examples are down-weighted.

    Dice loss: DL = 1 - (2 * intersection + eps) / (pred + target + eps)
    Optimizes for overlap between prediction and ground truth, which
    helps with class imbalance.

    Optional per-sample region weights are applied to the focal term:
    w = 1.0 + 2.0 * (water_ratio - min) / (max - min), normalized to
    mean 1 and clipped to [0.5, 2.5], giving more weight to samples
    from regions with more flood water.
    """
    def __init__(self, num_classes=2, focal_alpha=0.75, focal_gamma=2.0,
                 dice_weight=0.6, ignore_index=255):
        super().__init__()
        self.num_classes = num_classes
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.dice_weight = dice_weight
        self.ce_weight = 1.0 - dice_weight
        self.ignore_index = ignore_index
        self.class_weights = torch.tensor([1.0 - focal_alpha, focal_alpha], dtype=torch.float32)

    def forward(self, logits, target, sample_weights=None):
        B, C, H, W = logits.shape
        device = logits.device
        class_weights = self.class_weights.to(device)
        mask = (target != self.ignore_index)

        ce = F.cross_entropy(logits, target, weight=class_weights, ignore_index=self.ignore_index, reduction='none')
        ce = torch.where(mask, ce, torch.zeros_like(ce))

        probs = F.softmax(logits, dim=1)
        tgt = torch.clamp(target, 0, C - 1)
        pt = probs.gather(1, tgt.unsqueeze(1)).squeeze(1).clamp_(1e-6, 1 - 1e-6)
        focal = (1 - pt) ** self.focal_gamma
        focal_ce = focal * ce

        mask_f = mask.float()
        dice_losses = []
        eps = 1e-6

        for b in range(B):
            if mask_f[b].sum() < 1:
                continue

            dice_c = 0.0
            for c in range(C):
                p = probs[b, c] * mask_f[b]
                t = (tgt[b] == c).float() * mask_f[b]
                inter = (p * t).sum()
                denom = p.sum() + t.sum()
                dice_c += 1 - (2 * inter + eps) / (denom + eps)

            dice_losses.append(dice_c / C)

        dice_loss = torch.stack(dice_losses).mean() if dice_losses else torch.tensor(0.0, device=device)

        focal_per_sample = []
        for b in range(B):
            denom = mask_f[b].sum()
            if denom < 1:
                focal_per_sample.append(torch.tensor(0.0, device=device))
            else:
                focal_per_sample.append(focal_ce[b].sum() / denom)
        focal_per_sample = torch.stack(focal_per_sample)

        if sample_weights is not None:
            w = sample_weights.to(device).float()
            w = w / (w.mean() + 1e-6)
            w = torch.clamp(w, 0.5, 2.5)
            focal_term = (w * focal_per_sample).mean()
        else:
            focal_term = focal_per_sample.mean()

        total = self.dice_weight * dice_loss + self.ce_weight * focal_term
        total = torch.nan_to_num(total, nan=0.0, posinf=1e3, neginf=1e3)

        parts = {'dice': dice_loss.detach(), 'focal': focal_term.detach()}
        return total, parts

In [8]:
def train_epoch(model, loader, loss_fn, optimizer, scheduler, metrics, config, epoch):
    """Run one training epoch with gradient clipping and mixed precision."""
    model.train()
    device = config['device']
    scaler = config.get('scaler', None)
    use_amp = bool(config.get('amp', True) and scaler is not None)

    running_loss = 0.0
    valid_batches = 0
    skipped_batches = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Train]")
    for batch_idx, (data, target, meta) in enumerate(pbar):
        data = data.to(device, non_blocking=True).float()
        target = target.to(device, non_blocking=True).long()

        sample_weights = None
        if config.get('use_region_weights', False) and 'region_weights' in config:
            if isinstance(meta, dict) and 'region_id' in meta:
                ids = meta['region_id']
                weights = [config['region_weights'].get(str(ids[i]), 1.0) for i in range(data.size(0))]
                sample_weights = torch.tensor(weights, device=device, dtype=torch.float32)

        optimizer.zero_grad(set_to_none=True)

        try:
            with autocast(enabled=use_amp):
                logits = model(data)
                if not torch.isfinite(logits).all():
                    raise FloatingPointError("Non-finite logits")
                loss, parts = loss_fn(logits, target, sample_weights)

            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite loss")

            if use_amp:
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.get('gradient_clip', 10.0))
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.get('gradient_clip', 10.0))
                optimizer.step()

            running_loss += loss.item()
            valid_batches += 1
            pbar.set_postfix({'loss': f"{running_loss/max(1,valid_batches):.4f}"})

        except FloatingPointError:
            skipped_batches += 1
            continue

    return {
        'epoch': epoch,
        'train/loss': running_loss / max(1, valid_batches),
        'train/valid_batches': valid_batches,
        'train/skipped_batches': skipped_batches,
    }, None


def validate_epoch(model, loader, loss_fn, metrics, config, epoch):
    """Run one validation epoch and return overall and per-region IoU."""
    model.eval()
    device = config['device']
    scaler = config.get('scaler', None)
    use_amp = bool(config.get('amp', True) and scaler is not None)

    running_loss = 0.0
    valid_batches = 0
    skipped_batches = 0
    region_ious = defaultdict(list)
    all_ious = []

    with torch.no_grad():
        for batch_idx, (data, target, meta) in enumerate(tqdm(loader, desc=f"Epoch {epoch} [Val]")):
            data = data.to(device, non_blocking=True).float()
            target = target.to(device, non_blocking=True).long()

            try:
                with autocast(enabled=use_amp):
                    logits = model(data)
                    loss, _ = loss_fn(logits, target, None)

                if not torch.isfinite(loss):
                    raise FloatingPointError("Non-finite val loss")

                running_loss += loss.item()
                valid_batches += 1

                ious = metrics.iou_per_class(logits, target)
                all_ious.append(ious)

                regions = meta['region_id'] if isinstance(meta, dict) else [m['region_id'] for m in meta]
                for rid in regions:
                    region_ious[str(rid)].append(ious)

            except FloatingPointError:
                skipped_batches += 1
                continue

    if valid_batches == 0:
        return {
            'epoch': epoch,
            'val/loss': 999.0,
            'val/mIoU': 0.0,
            'val/water_IoU': 0.0,
            'val/bg_IoU': 0.0,
            'val/valid_batches': 0,
            'val/skipped_batches': skipped_batches
        }, torch.zeros(config['num_classes']), {}

    avg_loss = running_loss / valid_batches
    epoch_iou = np.mean(all_ious, axis=0) if all_ious else np.zeros(config['num_classes'])

    per_region_iou = {}
    for region_id, iou_list in region_ious.items():
        if len(iou_list) > 0:
            region_iou = np.mean(iou_list, axis=0)
            per_region_iou[region_id] = {
                'mean_IoU': region_iou.mean(),
                'water_IoU': region_iou[1],
                'bg_IoU': region_iou[0],
                'samples': len(iou_list)
            }

    return {
        'epoch': epoch,
        'val/loss': avg_loss,
        'val/mIoU': float(epoch_iou.mean()),
        'val/water_IoU': float(epoch_iou[1]),
        'val/bg_IoU': float(epoch_iou[0]),
        'val/valid_batches': valid_batches,
        'val/skipped_batches': skipped_batches
    }, epoch_iou, per_region_iou


def compute_region_weights(stats_df):
    """
    Compute a per-region sample weight from the water ratio.
    w = 1.0 + 2.0 * (ratio - min) / (max - min), so regions with more
    flood water get weight closer to 3.0 and the driest region gets 1.0.
    """
    train_stats = stats_df[stats_df['split'] == 'train']
    max_ratio = train_stats['water_ratio'].max()
    min_ratio = train_stats['water_ratio'].min()

    weights = {}
    for _, row in train_stats.iterrows():
        ratio = row['water_ratio']
        weight = 1.0 + 2.0 * (ratio - min_ratio) / (max_ratio - min_ratio + 1e-8)
        weights[row['region_id']] = weight

    return weights

In [9]:
TRAIN_CONFIG = {
    'num_classes': 2,
    'pretrained': True,
    'backbone': 'nvidia/mit-b2',
    'dropout': 0.1,

    'epochs': 200,
    'batch_size': 8,
    'learning_rate': 5e-5,
    'weight_decay': 1e-4,
    'min_lr': 1e-7,

    'focal_alpha': 0.75,
    'focal_gamma': 2.0,
    'dice_weight': 0.6,

    'warmup_epochs': 10,
    'gradient_clip': 10.0,
    'amp': True,

    'use_region_weights': True,

    'val_freq': 1,
    'save_freq': 10,
    'early_stopping_patience': 50,

    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

TRAIN_CONFIG['scaler'] = GradScaler(enabled=TRAIN_CONFIG['amp'])


def create_optimizer_scheduler(model, config):
    """Build the AdamW optimizer and a warmup plus cosine annealing LR scheduler."""
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay'],
        eps=1e-8
    )

    main_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config['epochs'] - config['warmup_epochs'],
        eta_min=config['min_lr']
    )

    if config['warmup_epochs'] > 0:
        warmup_scheduler = optim.lr_scheduler.LinearLR(
            optimizer,
            start_factor=0.1,
            total_iters=config['warmup_epochs']
        )
        scheduler = optim.lr_scheduler.SequentialLR(
            optimizer,
            [warmup_scheduler, main_scheduler],
            milestones=[config['warmup_epochs']]
        )
    else:
        scheduler = main_scheduler

    return optimizer, scheduler


def get_model_by_name(name, pretrained=True, num_classes=2, backbone='nvidia/mit-b2'):
    """Create a model instance from one of the entries in MODEL_CONFIGS."""
    if name not in MODEL_CONFIGS:
        raise ValueError(f"Unknown model name: {name}. Available: {list(MODEL_CONFIGS.keys())}")

    cfg = MODEL_CONFIGS[name]
    return create_model(
        model_type=cfg['model_type'],
        backbone=backbone,
        fusion_type=cfg['fusion_type'],
        pretrained=pretrained,
        num_classes=num_classes
    )

/tmp/ipykernel_4913/868314042.py:30: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  TRAIN_CONFIG['scaler'] = GradScaler(enabled=TRAIN_CONFIG['amp'])


# MUST-Former-SAR

In [19]:
model_name = 's1_only'
config = TRAIN_CONFIG.copy()
config['model_name'] = model_name
config['scaler'] = GradScaler(enabled=config['amp'])

train_loader, val_loader, test_loader = create_dataloaders(
    train_df, val_df, test_df,
    batch_size=config['batch_size'],
    include_s1=True,
    include_s2=False
)

model = get_model_by_name(
    model_name,
    pretrained=config['pretrained'],
    num_classes=config['num_classes'],
    backbone=config['backbone']
).to(config['device'])

loss_fn = HybridLoss(
    num_classes=config['num_classes'],
    focal_alpha=config['focal_alpha'],
    focal_gamma=config['focal_gamma'],
    dice_weight=config['dice_weight']
).to(config['device'])

metrics = SegmentationMetrics(num_classes=config['num_classes'])
optimizer, scheduler = create_optimizer_scheduler(model, config)

stats_path = os.path.join(paths.OUTPUT_DIR, 'region_water_stats.csv')
region_stats_df = pd.read_csv(stats_path)
config['region_weights'] = compute_region_weights(region_stats_df)

print("Region weights:")
for region_id, weight in sorted(config['region_weights'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {region_id}: {weight:.4f}")

best_val_iou = 0.0
patience_counter = 0
train_history = []
val_history = []

print(f"Starting training for {config['epochs']} epochs")

for epoch in range(1, config['epochs'] + 1):
    train_log, _ = train_epoch(model, train_loader, loss_fn, optimizer, scheduler, metrics, config, epoch)
    train_history.append(train_log)

    if epoch % config['val_freq'] == 0:
        val_log, val_iou, per_region_iou = validate_epoch(model, val_loader, loss_fn, metrics, config, epoch)
        val_history.append(val_log)

        print(f"Epoch {epoch}/{config['epochs']}")
        print(f"  Train Loss: {train_log['train/loss']:.4f} | Skip: {train_log['train/skipped_batches']}")
        print(f"  Val Loss: {val_log['val/loss']:.4f} | mIoU: {val_log['val/mIoU']:.4f} | Water IoU: {val_log['val/water_IoU']:.4f}")

        if val_log['val/water_IoU'] > best_val_iou:
            best_val_iou = val_log['val/water_IoU']
            patience_counter = 0

            checkpoint_path = os.path.join(paths.OUTPUT_DIR, 'checkpoints', f'{model_name}_best.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_iou': val_iou,
                'config': config
            }, checkpoint_path)
            print(f"  Saved best model (Water IoU: {best_val_iou:.4f})")
        else:
            patience_counter += 1

        if patience_counter >= config['early_stopping_patience']:
            print(f"Early stopping at epoch {epoch}")
            break

    scheduler.step()

    if epoch % config['save_freq'] == 0:
        checkpoint_path = os.path.join(paths.OUTPUT_DIR, 'checkpoints', f'{model_name}_epoch{epoch}.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': config
        }, checkpoint_path)

history_path = os.path.join(paths.OUTPUT_DIR, f'{model_name}_history.json')
with open(history_path, 'w') as f:
    json.dump({'train': train_history, 'val': val_history}, f, indent=2)

print(f"Training complete: {model_name}")
print(f"Best Water IoU: {best_val_iou:.4f}")

del model, optimizer, scheduler, loss_fn
gc.collect()
torch.cuda.empty_cache()

# MUST-Former-Optical

In [18]:
model_name = 's2_only'
config = TRAIN_CONFIG.copy()
config['model_name'] = model_name
config['scaler'] = GradScaler(enabled=config['amp'])

train_loader, val_loader, test_loader = create_dataloaders(
    train_df, val_df, test_df,
    batch_size=config['batch_size'],
    include_s1=False,
    include_s2=True
)

model = get_model_by_name(
    model_name,
    pretrained=config['pretrained'],
    num_classes=config['num_classes'],
    backbone=config['backbone']
).to(config['device'])

loss_fn = HybridLoss(
    num_classes=config['num_classes'],
    focal_alpha=config['focal_alpha'],
    focal_gamma=config['focal_gamma'],
    dice_weight=config['dice_weight']
).to(config['device'])

metrics = SegmentationMetrics(num_classes=config['num_classes'])
optimizer, scheduler = create_optimizer_scheduler(model, config)

stats_path = os.path.join(paths.OUTPUT_DIR, 'region_water_stats.csv')
region_stats_df = pd.read_csv(stats_path)
config['region_weights'] = compute_region_weights(region_stats_df)

print("Region weights:")
for region_id, weight in sorted(config['region_weights'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {region_id}: {weight:.4f}")

best_val_iou = 0.0
patience_counter = 0
train_history = []
val_history = []

print(f"Starting training for {config['epochs']} epochs")

for epoch in range(1, config['epochs'] + 1):
    train_log, _ = train_epoch(model, train_loader, loss_fn, optimizer, scheduler, metrics, config, epoch)
    train_history.append(train_log)

    if epoch % config['val_freq'] == 0:
        val_log, val_iou, per_region_iou = validate_epoch(model, val_loader, loss_fn, metrics, config, epoch)
        val_history.append(val_log)

        print(f"Epoch {epoch}/{config['epochs']}")
        print(f"  Train Loss: {train_log['train/loss']:.4f} | Skip: {train_log['train/skipped_batches']}")
        print(f"  Val Loss: {val_log['val/loss']:.4f} | mIoU: {val_log['val/mIoU']:.4f} | Water IoU: {val_log['val/water_IoU']:.4f}")

        if val_log['val/water_IoU'] > best_val_iou:
            best_val_iou = val_log['val/water_IoU']
            patience_counter = 0

            checkpoint_path = os.path.join(paths.OUTPUT_DIR, 'checkpoints', f'{model_name}_best.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_iou': val_iou,
                'config': config
            }, checkpoint_path)
            print(f"  Saved best model (Water IoU: {best_val_iou:.4f})")
        else:
            patience_counter += 1

        if patience_counter >= config['early_stopping_patience']:
            print(f"Early stopping at epoch {epoch}")
            break

    scheduler.step()

    if epoch % config['save_freq'] == 0:
        checkpoint_path = os.path.join(paths.OUTPUT_DIR, 'checkpoints', f'{model_name}_epoch{epoch}.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': config
        }, checkpoint_path)

history_path = os.path.join(paths.OUTPUT_DIR, f'{model_name}_history.json')
with open(history_path, 'w') as f:
    json.dump({'train': train_history, 'val': val_history}, f, indent=2)

print(f"Training complete: {model_name}")
print(f"Best Water IoU: {best_val_iou:.4f}")

del model, optimizer, scheduler, loss_fn
gc.collect()
torch.cuda.empty_cache()

# MUST-Former-Projector

In [17]:
import gc

model_name = 'fusion_projector'
config = TRAIN_CONFIG.copy()
config['model_name'] = model_name
config['scaler'] = GradScaler(enabled=config['amp'])

train_loader, val_loader, test_loader = create_dataloaders(
    train_df, val_df, test_df,
    batch_size=config['batch_size'],
    include_s1=True,
    include_s2=True
)

model = get_model_by_name(
    model_name,
    pretrained=config['pretrained'],
    num_classes=config['num_classes'],
    backbone=config['backbone']
).to(config['device'])

loss_fn = HybridLoss(
    num_classes=config['num_classes'],
    focal_alpha=config['focal_alpha'],
    focal_gamma=config['focal_gamma'],
    dice_weight=config['dice_weight']
).to(config['device'])

metrics = SegmentationMetrics(num_classes=config['num_classes'])
optimizer, scheduler = create_optimizer_scheduler(model, config)

stats_path = os.path.join(paths.OUTPUT_DIR, 'region_water_stats.csv')
region_stats_df = pd.read_csv(stats_path)
config['region_weights'] = compute_region_weights(region_stats_df)

print("Region weights:")
for region_id, weight in sorted(config['region_weights'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {region_id}: {weight:.4f}")

best_val_iou = 0.0
patience_counter = 0
train_history = []
val_history = []

print(f"Starting training for {config['epochs']} epochs")

for epoch in range(1, config['epochs'] + 1):
    train_log, _ = train_epoch(model, train_loader, loss_fn, optimizer, scheduler, metrics, config, epoch)
    train_history.append(train_log)

    if epoch % config['val_freq'] == 0:
        val_log, val_iou, per_region_iou = validate_epoch(model, val_loader, loss_fn, metrics, config, epoch)
        val_history.append(val_log)

        print(f"Epoch {epoch}/{config['epochs']}")
        print(f"  Train Loss: {train_log['train/loss']:.4f} | Skip: {train_log['train/skipped_batches']}")
        print(f"  Val Loss: {val_log['val/loss']:.4f} | mIoU: {val_log['val/mIoU']:.4f} | Water IoU: {val_log['val/water_IoU']:.4f}")

        if val_log['val/water_IoU'] > best_val_iou:
            best_val_iou = val_log['val/water_IoU']
            patience_counter = 0

            checkpoint_path = os.path.join(paths.OUTPUT_DIR, 'checkpoints', f'{model_name}_best.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_iou': val_iou,
                'config': config
            }, checkpoint_path)
            print(f"  Saved best model (Water IoU: {best_val_iou:.4f})")
        else:
            patience_counter += 1

        if patience_counter >= config['early_stopping_patience']:
            print(f"Early stopping at epoch {epoch}")
            break

    scheduler.step()

    if epoch % config['save_freq'] == 0:
        checkpoint_path = os.path.join(paths.OUTPUT_DIR, 'checkpoints', f'{model_name}_epoch{epoch}.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': config
        }, checkpoint_path)

history_path = os.path.join(paths.OUTPUT_DIR, f'{model_name}_history.json')
with open(history_path, 'w') as f:
    json.dump({'train': train_history, 'val': val_history}, f, indent=2)

print(f"Training complete: {model_name}")
print(f"Best Water IoU: {best_val_iou:.4f}")

del model, optimizer, scheduler, loss_fn
gc.collect()
torch.cuda.empty_cache()

# MUST-Former-CrossAttn

In [16]:
model_name = 'fusion_attention'
config = TRAIN_CONFIG.copy()
config['model_name'] = model_name
config['scaler'] = GradScaler(enabled=config['amp'])
config['batch_size'] = 4

train_loader, val_loader, test_loader = create_dataloaders(
    train_df, val_df, test_df,
    batch_size=config['batch_size'],
    include_s1=True,
    include_s2=True
)

model = get_model_by_name(
    model_name,
    pretrained=config['pretrained'],
    num_classes=config['num_classes'],
    backbone=config['backbone']
).to(config['device'])

loss_fn = HybridLoss(
    num_classes=config['num_classes'],
    focal_alpha=config['focal_alpha'],
    focal_gamma=config['focal_gamma'],
    dice_weight=config['dice_weight']
).to(config['device'])

metrics = SegmentationMetrics(num_classes=config['num_classes'])
optimizer, scheduler = create_optimizer_scheduler(model, config)

stats_path = os.path.join(paths.OUTPUT_DIR, 'region_water_stats.csv')
region_stats_df = pd.read_csv(stats_path)
config['region_weights'] = compute_region_weights(region_stats_df)

print("Region weights:")
for region_id, weight in sorted(config['region_weights'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {region_id}: {weight:.4f}")

best_val_iou = 0.0
patience_counter = 0
train_history = []
val_history = []

print(f"Starting training for {config['epochs']} epochs")

for epoch in range(1, config['epochs'] + 1):
    train_log, _ = train_epoch(model, train_loader, loss_fn, optimizer, scheduler, metrics, config, epoch)
    train_history.append(train_log)

    if epoch % config['val_freq'] == 0:
        val_log, val_iou, per_region_iou = validate_epoch(model, val_loader, loss_fn, metrics, config, epoch)
        val_history.append(val_log)

        print(f"Epoch {epoch}/{config['epochs']}")
        print(f"  Train Loss: {train_log['train/loss']:.4f} | Skip: {train_log['train/skipped_batches']}")
        print(f"  Val Loss: {val_log['val/loss']:.4f} | mIoU: {val_log['val/mIoU']:.4f} | Water IoU: {val_log['val/water_IoU']:.4f}")

        if val_log['val/water_IoU'] > best_val_iou:
            best_val_iou = val_log['val/water_IoU']
            patience_counter = 0

            checkpoint_path = os.path.join(paths.OUTPUT_DIR, 'checkpoints', f'{model_name}_best.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_iou': val_iou,
                'config': config
            }, checkpoint_path)
            print(f"  Saved best model (Water IoU: {best_val_iou:.4f})")
        else:
            patience_counter += 1

        if patience_counter >= config['early_stopping_patience']:
            print(f"Early stopping at epoch {epoch}")
            break

    scheduler.step()

    if epoch % config['save_freq'] == 0:
        checkpoint_path = os.path.join(paths.OUTPUT_DIR, 'checkpoints', f'{model_name}_epoch{epoch}.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': config
        }, checkpoint_path)

history_path = os.path.join(paths.OUTPUT_DIR, f'{model_name}_history.json')
with open(history_path, 'w') as f:
    json.dump({'train': train_history, 'val': val_history}, f, indent=2)

print(f"Training complete: {model_name}")
print(f"Best Water IoU: {best_val_iou:.4f}")

del model, optimizer, scheduler, loss_fn
gc.collect()
torch.cuda.empty_cache()

# MUST-Former-PCA

In [15]:
model_name = 'fusion_pca'
config = TRAIN_CONFIG.copy()
config['model_name'] = model_name
config['scaler'] = GradScaler(enabled=config['amp'])

train_loader, val_loader, test_loader = create_dataloaders(
    train_df, val_df, test_df,
    batch_size=config['batch_size'],
    include_s1=True,
    include_s2=True
)

model = get_model_by_name(
    model_name,
    pretrained=config['pretrained'],
    num_classes=config['num_classes'],
    backbone=config['backbone']
).to(config['device'])

loss_fn = HybridLoss(
    num_classes=config['num_classes'],
    focal_alpha=config['focal_alpha'],
    focal_gamma=config['focal_gamma'],
    dice_weight=config['dice_weight']
).to(config['device'])

metrics = SegmentationMetrics(num_classes=config['num_classes'])
optimizer, scheduler = create_optimizer_scheduler(model, config)

stats_path = os.path.join(paths.OUTPUT_DIR, 'region_water_stats.csv')
region_stats_df = pd.read_csv(stats_path)
config['region_weights'] = compute_region_weights(region_stats_df)

print("Region weights:")
for region_id, weight in sorted(config['region_weights'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {region_id}: {weight:.4f}")

best_val_iou = 0.0
patience_counter = 0
train_history = []
val_history = []

print(f"Starting training for {config['epochs']} epochs")

for epoch in range(1, config['epochs'] + 1):
    train_log, _ = train_epoch(model, train_loader, loss_fn, optimizer, scheduler, metrics, config, epoch)
    train_history.append(train_log)

    if epoch % config['val_freq'] == 0:
        val_log, val_iou, per_region_iou = validate_epoch(model, val_loader, loss_fn, metrics, config, epoch)
        val_history.append(val_log)

        print(f"Epoch {epoch}/{config['epochs']}")
        print(f"  Train Loss: {train_log['train/loss']:.4f} | Skip: {train_log['train/skipped_batches']}")
        print(f"  Val Loss: {val_log['val/loss']:.4f} | mIoU: {val_log['val/mIoU']:.4f} | Water IoU: {val_log['val/water_IoU']:.4f}")

        if val_log['val/water_IoU'] > best_val_iou:
            best_val_iou = val_log['val/water_IoU']
            patience_counter = 0

            checkpoint_path = os.path.join(paths.OUTPUT_DIR, 'checkpoints', f'{model_name}_best.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_iou': val_iou,
                'config': config
            }, checkpoint_path)
            print(f"  Saved best model (Water IoU: {best_val_iou:.4f})")
        else:
            patience_counter += 1

        if patience_counter >= config['early_stopping_patience']:
            print(f"Early stopping at epoch {epoch}")
            break

    scheduler.step()

    if epoch % config['save_freq'] == 0:
        checkpoint_path = os.path.join(paths.OUTPUT_DIR, 'checkpoints', f'{model_name}_epoch{epoch}.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': config
        }, checkpoint_path)

history_path = os.path.join(paths.OUTPUT_DIR, f'{model_name}_history.json')
with open(history_path, 'w') as f:
    json.dump({'train': train_history, 'val': val_history}, f, indent=2)

print(f"Training complete: {model_name}")
print(f"Best Water IoU: {best_val_iou:.4f}")

del model, optimizer, scheduler, loss_fn
gc.collect()
torch.cuda.empty_cache()